# SQL 性能调优 (Query Performance Tuning)

> **适用场景**: 慢查询优化、大表处理、数据仓库性能工程
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频（Senior DE 必考）

## 目录

1. 执行计划 EXPLAIN / EXPLAIN ANALYZE
2. 索引类型：B-Tree / Hash / Bitmap / Composite
3. 分区表 & 分区裁剪
4. Statistics 更新 & Cardinality 估算
5. 物化视图 vs 普通视图
6. 练习题

In [ ]:
# !pip install duckdb -q
import duckdb
import pandas as pd
import time

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

In [ ]:
# 创建较大的示例数据以观察性能差异
con.execute("""
CREATE OR REPLACE TABLE large_orders AS
SELECT
    row_number() OVER () AS order_id,
    (random() * 10000)::INTEGER AS customer_id,
    (random() * 5)::INTEGER + 1 AS product_category_id,  -- 1-5
    DATE '2020-01-01' + (random() * 1460)::INTEGER AS order_date,  -- 4年
    CASE (random() * 4)::INTEGER
        WHEN 0 THEN 'North'
        WHEN 1 THEN 'South'
        WHEN 2 THEN 'East'
        ELSE         'West'
    END AS region,
    ROUND((random() * 5000 + 10)::NUMERIC, 2) AS amount,
    CASE (random() * 3)::INTEGER
        WHEN 0 THEN 'completed'
        WHEN 1 THEN 'pending'
        ELSE        'cancelled'
    END AS status
FROM generate_series(1, 500000)  -- 50 万条记录
""")

con.execute("""
CREATE OR REPLACE TABLE products AS
SELECT * FROM (VALUES
    (1, 'Electronics', 'High'),
    (2, 'Clothing',    'Medium'),
    (3, 'Food',        'Low'),
    (4, 'Furniture',   'High'),
    (5, 'Books',       'Low')
) t(category_id, category_name, margin_tier)
""")

row_count = con.execute("SELECT COUNT(*) FROM large_orders").fetchone()[0]
print(f"large_orders 表已创建，共 {row_count:,} 条记录")
con.execute("SELECT * FROM large_orders LIMIT 5").df()

---

## 1. 执行计划 EXPLAIN / EXPLAIN ANALYZE

执行计划是数据库优化查询的核心工具，揭示数据库**如何执行**一条 SQL 语句。

### 关键概念

```
EXPLAIN          — 显示计划（不实际执行）
EXPLAIN ANALYZE  — 执行查询并显示实际运行时间（PostgreSQL/DuckDB 支持）
```

### 执行计划中的关键指标

```
┌─────────────────────────────────────────────────┐
│          执行计划常见节点                          │
├─────────────────────────────────────────────────┤
│  Seq Scan      - 全表扫描（无索引或优化器选择）    │
│  Index Scan    - 使用索引扫描（适合高选择性查询）  │
│  Bitmap Scan   - 位图索引扫描（适合低选择性查询）  │
│  Hash Join     - 散列连接（适合大表等值连接）      │
│  Merge Join    - 归并连接（适合已排序数据）        │
│  Nested Loop   - 嵌套循环（适合小表或有索引）      │
│  Sort          - 排序操作                         │
│  Aggregate     - 聚合操作                         │
├─────────────────────────────────────────────────┤
│  cost=X..Y     - 预估代价（启动..总计）           │
│  rows=N        - 预估返回行数                     │
│  actual time=X..Y  - 实际耗时（ms）               │
│  actual rows=N     - 实际返回行数                 │
│  loops=N           - 该节点执行次数               │
└─────────────────────────────────────────────────┘
```

**调优核心**: 比较 `rows=估算值` 和 `actual rows=实际值`，差异过大说明统计信息过时。

In [ ]:
# EXPLAIN：查看基础查询计划
print("=== 简单查询的执行计划 ===")
plan = con.execute("""
EXPLAIN
SELECT region, SUM(amount) as total
FROM large_orders
WHERE status = 'completed'
GROUP BY region
ORDER BY total DESC
""").fetchall()

for row in plan:
    print(row[0])

In [ ]:
# EXPLAIN ANALYZE：实际执行并报告时间
print("=== EXPLAIN ANALYZE：JOIN 查询 ===")
plan = con.execute("""
EXPLAIN ANALYZE
SELECT
    p.category_name,
    o.region,
    COUNT(*) AS order_count,
    SUM(o.amount) AS total_revenue
FROM large_orders o
JOIN products p ON o.product_category_id = p.category_id
WHERE o.order_date >= DATE '2022-01-01'
  AND o.status = 'completed'
GROUP BY p.category_name, o.region
ORDER BY total_revenue DESC
""").fetchall()

for row in plan:
    print(row[0])

In [ ]:
# 测量查询执行时间（DuckDB 方式）
queries = {
    "全表扫描 (无过滤)": "SELECT COUNT(*) FROM large_orders",
    "带 WHERE 过滤": "SELECT COUNT(*) FROM large_orders WHERE status = 'completed'",
    "GROUP BY 聚合": "SELECT region, COUNT(*), SUM(amount) FROM large_orders GROUP BY region",
    "JOIN 查询": """
        SELECT p.category_name, COUNT(*)
        FROM large_orders o
        JOIN products p ON o.product_category_id = p.category_id
        GROUP BY p.category_name
    """
}

for name, sql in queries.items():
    start = time.time()
    con.execute(sql).fetchall()
    elapsed = (time.time() - start) * 1000
    print(f"{name}: {elapsed:.2f} ms")

---

## 2. 索引类型：B-Tree / Hash / Bitmap / Composite

索引是提升查询性能的核心机制，但也有写入开销和存储成本。

```
┌──────────────────────────────────────────────────────────────────────┐
│                     索引类型对比                                       │
├──────────────┬───────────────┬────────────────┬──────────────────────┤
│   类型        │  数据结构      │  适用场景       │  不适用场景           │
├──────────────┼───────────────┼────────────────┼──────────────────────┤
│  B-Tree      │ 平衡树         │ 范围查询、=、   │ 向量/文本相似度查询   │
│  (默认)       │               │ <、>、ORDER BY │                      │
├──────────────┼───────────────┼────────────────┼──────────────────────┤
│  Hash        │ 哈希表         │ 只有 = 等值查询 │ 范围查询             │
│              │               │ 极快点查        │ ORDER BY、LIKE       │
├──────────────┼───────────────┼────────────────┼──────────────────────┤
│  Bitmap      │ 位图           │ 低基数列        │ 高并发写入（写锁竞争）│
│              │               │ (status/region) │ 高基数列（效率低）   │
├──────────────┼───────────────┼────────────────┼──────────────────────┤
│  Composite   │ 多列 B-Tree   │ 多列联合查询    │ 前缀列未在 WHERE 中  │
│  (复合索引)   │               │ 覆盖索引        │                      │
└──────────────┴───────────────┴────────────────┴──────────────────────┘
```

### 索引选择性（Selectivity）

```
选择性 = 唯一值数量 / 总行数
高选择性（接近1.0）→ 适合 B-Tree 索引 → 精确定位少量行
低选择性（接近0.0）→ 考虑 Bitmap 索引 → 如 status(3个值)/性别(2个值)
```

### 复合索引最左前缀原则

```
INDEX(a, b, c) 可以支持：
  WHERE a = ?          ✓ 使用索引
  WHERE a = ? AND b = ?  ✓ 使用索引
  WHERE a = ? AND b = ? AND c = ?  ✓ 使用完整索引
  WHERE b = ?          ✗ 无法使用（跳过了 a）
  WHERE a = ? AND c = ?  ✓ 部分使用（仅 a 列）
```

In [ ]:
# 分析各列的选择性
selectivity = con.execute("""
SELECT
    '  order_id'            AS column_name,
    COUNT(DISTINCT order_id)            AS distinct_values,
    COUNT(*)                            AS total_rows,
    ROUND(COUNT(DISTINCT order_id) * 1.0 / COUNT(*), 4) AS selectivity,
    CASE
        WHEN COUNT(DISTINCT order_id) * 1.0 / COUNT(*) > 0.8 THEN 'High → B-Tree'
        WHEN COUNT(DISTINCT order_id) * 1.0 / COUNT(*) > 0.1 THEN 'Medium → B-Tree'
        ELSE 'Low → Bitmap'
    END AS recommended_index
FROM large_orders
UNION ALL
SELECT 'customer_id', COUNT(DISTINCT customer_id), COUNT(*),
       ROUND(COUNT(DISTINCT customer_id) * 1.0 / COUNT(*), 4),
       CASE WHEN COUNT(DISTINCT customer_id) * 1.0 / COUNT(*) > 0.8 THEN 'High → B-Tree'
            WHEN COUNT(DISTINCT customer_id) * 1.0 / COUNT(*) > 0.1 THEN 'Medium → B-Tree'
            ELSE 'Low → Bitmap' END
FROM large_orders
UNION ALL
SELECT 'region', COUNT(DISTINCT region), COUNT(*),
       ROUND(COUNT(DISTINCT region) * 1.0 / COUNT(*), 4),
       CASE WHEN COUNT(DISTINCT region) * 1.0 / COUNT(*) > 0.1 THEN 'Medium → B-Tree'
            ELSE 'Low → Bitmap' END
FROM large_orders
UNION ALL
SELECT 'status', COUNT(DISTINCT status), COUNT(*),
       ROUND(COUNT(DISTINCT status) * 1.0 / COUNT(*), 4),
       'Low → Bitmap'
FROM large_orders
""").df()

print("各列选择性分析（指导索引选型）:")
selectivity

In [ ]:
# 覆盖索引（Covering Index）演示
# 覆盖索引：查询所需的所有列都在索引中，无需回表
print("""
覆盖索引概念说明:
================
普通索引查询流程:
  SQL → 索引查找 order_date → 得到 rowid → 回表读取 amount, status → 返回
           ↑                                    ↑
        快速（索引）                          慢（随机IO）

覆盖索引查询流程（order_date, amount, status 都在索引中）:
  SQL → 索引查找 order_date → 直接从索引读取 amount, status → 返回
           ↑
    只需索引IO，无需回表

PostgreSQL 建覆盖索引:
  CREATE INDEX idx_covering ON large_orders (order_date)
    INCLUDE (amount, status);
""")

# 用执行计划模拟覆盖索引效果
plan = con.execute("""
EXPLAIN
SELECT order_date, amount, status
FROM large_orders
WHERE order_date BETWEEN DATE '2023-01-01' AND DATE '2023-12-31'
""").fetchall()
for row in plan:
    print(row[0])

In [ ]:
# 索引失效场景演示（重要！）
print("""
索引失效场景（常见面试考点）:
==============================

1. 对索引列使用函数:
   ❌ WHERE YEAR(order_date) = 2023        -- 函数包裹导致无法用索引
   ✓  WHERE order_date BETWEEN '2023-01-01' AND '2023-12-31'

2. 隐式类型转换:
   ❌ WHERE customer_id = '1001'           -- 字符串 vs 整数，触发转换
   ✓  WHERE customer_id = 1001

3. 前缀通配符:
   ❌ WHERE status LIKE '%cancel%'         -- 前缀 % 无法用 B-Tree 索引
   ✓  WHERE status LIKE 'cancel%'          -- 前缀固定可以用索引

4. OR 条件跨列:
   ❌ WHERE region = 'North' OR status = 'completed'  -- 可能导致全表扫描
   ✓  使用 UNION ALL 拆分为两个查询

5. 不等号 + 复合索引后续列:
   INDEX(a, b) → WHERE a > 5 AND b = 'X'
   ❌ b 列无法使用索引（a 是范围查询后 b 的顺序混乱）
""")

---

## 3. 分区表 & 分区裁剪

分区是将一张逻辑大表按某个策略分割成多个物理子表，每个子表是一个**分区**。

```
分区策略:
┌─────────────────────────────────────────────────────────┐
│  Range 分区  │ 按值范围划分 (时间、ID范围)               │
│              │ 最常用！数仓通常按日期分区                │
├─────────────────────────────────────────────────────────┤
│  List 分区   │ 按枚举值划分 (地区、状态、类别)           │
│              │ 适合低基数列                              │
├─────────────────────────────────────────────────────────┤
│  Hash 分区   │ 按哈希值均匀分布 (高基数ID)              │
│              │ 解决数据倾斜问题                          │
├─────────────────────────────────────────────────────────┤
│  复合分区    │ 先按时间分区，再按地区子分区              │
│              │ 大规模数仓常用                            │
└─────────────────────────────────────────────────────────┘
```

### 分区裁剪（Partition Pruning）

当 WHERE 条件包含分区键时，优化器会**跳过不相关的分区**，只扫描必要的分区。

```sql
-- 按年份分区的表，WHERE 包含分区键时只扫描 2023 分区
SELECT * FROM orders WHERE order_date >= '2023-01-01'
-- 优化器看到：只需扫描 orders_2023 分区，跳过 2020/2021/2022
```

In [ ]:
# DuckDB 中用视图模拟分区概念（DuckDB 原生不支持 DDL 分区，但支持分区文件）
# 这里演示分区裁剪的核心思想

# 创建按年份"分区"的数据（模拟 Hive/Iceberg 分区文件）
for year in [2020, 2021, 2022, 2023]:
    con.execute(f"""
    CREATE OR REPLACE TABLE orders_{year} AS
    SELECT *
    FROM large_orders
    WHERE YEAR(order_date) = {year}
    """)
    count = con.execute(f"SELECT COUNT(*) FROM orders_{year}").fetchone()[0]
    print(f"orders_{year}: {count:,} 行")

# 创建 UNION ALL 视图（类似分区表的逻辑视图）
con.execute("""
CREATE OR REPLACE VIEW partitioned_orders AS
SELECT *, 2020 AS partition_year FROM orders_2020
UNION ALL
SELECT *, 2021 AS partition_year FROM orders_2021
UNION ALL
SELECT *, 2022 AS partition_year FROM orders_2022
UNION ALL
SELECT *, 2023 AS partition_year FROM orders_2023
""")
print("\n分区视图创建完成")

In [ ]:
# 演示分区裁剪效果
print("方法 1：直接查询特定年份分区（最高效）")
start = time.time()
result = con.execute("""
SELECT COUNT(*), SUM(amount) FROM orders_2023
WHERE status = 'completed'
""").fetchone()
t1 = (time.time() - start) * 1000
print(f"  结果: {result[0]:,} 订单, 总金额 {result[1]:,.2f}")
print(f"  耗时: {t1:.2f} ms")

print("\n方法 2：通过 UNION ALL 视图（带分区裁剪条件）")
start = time.time()
result = con.execute("""
SELECT COUNT(*), SUM(amount) FROM partitioned_orders
WHERE partition_year = 2023  -- 分区裁剪条件
  AND status = 'completed'
""").fetchone()
t2 = (time.time() - start) * 1000
print(f"  结果: {result[0]:,} 订单, 总金额 {result[1]:,.2f}")
print(f"  耗时: {t2:.2f} ms")

print("\n方法 3：全表扫描（无分区裁剪，最慢）")
start = time.time()
result = con.execute("""
SELECT COUNT(*), SUM(amount) FROM large_orders
WHERE YEAR(order_date) = 2023  -- 无法利用分区裁剪
  AND status = 'completed'
""").fetchone()
t3 = (time.time() - start) * 1000
print(f"  结果: {result[0]:,} 订单, 总金额 {result[1]:,.2f}")
print(f"  耗时: {t3:.2f} ms")

In [ ]:
# PostgreSQL 风格的分区表 DDL（参考代码，无法在 DuckDB 执行）
print("""
PostgreSQL 分区表 DDL 参考:
===========================

-- 创建分区父表
CREATE TABLE orders (
    order_id    BIGINT,
    order_date  DATE NOT NULL,
    customer_id INT,
    amount      NUMERIC(12,2),
    status      VARCHAR(20)
) PARTITION BY RANGE (order_date);  -- 按日期范围分区

-- 创建各年份分区
CREATE TABLE orders_2022 PARTITION OF orders
    FOR VALUES FROM ('2022-01-01') TO ('2023-01-01');

CREATE TABLE orders_2023 PARTITION OF orders
    FOR VALUES FROM ('2023-01-01') TO ('2024-01-01');

-- 列表分区示例（按地区）
CREATE TABLE orders_north PARTITION OF orders_by_region
    FOR VALUES IN ('North', 'NorthEast', 'NorthWest');

-- 查询时自动分区裁剪（不需要指定分区名）
SELECT * FROM orders
WHERE order_date >= '2023-01-01';  -- 优化器自动只扫描 orders_2023

-- 分区注意事项:
-- 1. 分区键应是查询中的高频过滤条件
-- 2. 时间序列数据首选 Range 分区
-- 3. 分区键参与 JOIN 条件时可触发分区 JOIN 优化
-- 4. 需要定期创建新分区（可用存储过程自动化）
""")

---

## 4. Statistics 更新 & Cardinality 估算

### 统计信息的作用

数据库优化器依赖统计信息来**估算查询的代价和结果行数**，从而选择最优执行计划。

```
统计信息包含:
┌────────────────────────────────────────────────────┐
│  表级统计    │ 总行数、总页数                        │
│  列级统计    │ 唯一值数量（NDV）、NULL 比例           │
│              │ 最小/最大值、最常见值（MCV）          │
│              │ 直方图（值分布）                      │
│  相关性统计  │ 多列联合分布（PostgreSQL 扩展统计）    │
└────────────────────────────────────────────────────┘
```

### 统计信息过时的影响

```
实际情况:  表有 10,000,000 行
统计信息:  表有 1,000 行（大量数据导入后未更新统计）

优化器决策:
  以为数据少 → 选择 Nested Loop Join（适合小表）
  实际数据多 → Nested Loop 执行极慢！
  正确选择应是 → Hash Join
```

### PostgreSQL 统计信息命令

```sql
ANALYZE table_name;                -- 更新指定表统计信息
ANALYZE;                           -- 更新所有表统计信息
VACUUM ANALYZE table_name;         -- 回收空间并更新统计

-- 查看统计信息
SELECT * FROM pg_stats WHERE tablename = 'orders';
```

In [ ]:
# DuckDB 内置统计分析
print("=== DuckDB 的列统计信息 ===")
stats = con.execute("""
SUMMARIZE large_orders
""").df()
stats

In [ ]:
# Cardinality 估算对执行计划的影响
print("""
Cardinality（基数）估算误差的实际影响:
=======================================

场景：两表 JOIN，优化器估算 orders 过滤后只有 100 行

因估算不准:
  优化器选择 → Nested Loop Join（认为外表很小）
  实际行数   → 50,000 行
  实际效果   → 内表被扫描 50,000 次 → 极其缓慢

正确估算下:
  优化器选择 → Hash Join（构建哈希表一次，然后探测）
  实际效果   → 快速完成

诊断方法（PostgreSQL）:
  EXPLAIN ANALYZE 输出中比较:
    rows=100      ← 估算值（错误！）
    actual rows=50000  ← 实际值
  如果偏差 > 10x，需要 ANALYZE 更新统计信息
""")

# 在 DuckDB 中查看估算 vs 实际
plan = con.execute("""
EXPLAIN ANALYZE
SELECT region, COUNT(*)
FROM large_orders
WHERE amount > 4000  -- 高选择性过滤
GROUP BY region
""").fetchall()
for row in plan:
    print(row[0])

---

## 5. 物化视图 vs 普通视图

```
┌──────────────────────────────────────────────────────────────────┐
│              普通视图 vs 物化视图对比                              │
├──────────────────┬────────────────┬────────────────────────────── │
│  特性             │  普通视图 (VIEW) │  物化视图 (Materialized View) │
├──────────────────┼────────────────┼──────────────────────────────┤
│  数据存储         │  不存储数据     │  存储查询结果（物理表）        │
│  查询时行为       │  每次查询重新计算│  读取已缓存的结果             │
│  数据新鲜度       │  始终最新       │  需要刷新（手动或定时）        │
│  读取性能         │  依赖基础表     │  极快（已预计算）             │
│  写入影响         │  无            │  刷新时有写入开销             │
│  适用场景         │  简化复杂查询   │  报表加速、预计算聚合          │
│                  │  权限控制       │  降低实时计算压力             │
└──────────────────┴────────────────┴──────────────────────────────┘
```

### 物化视图刷新策略

```sql
-- PostgreSQL
REFRESH MATERIALIZED VIEW mv_name;                 -- 完全刷新（锁表）
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_name;    -- 并发刷新（需 UNIQUE 索引）

-- 增量刷新（某些数据库支持）
-- Snowflake、BigQuery 等支持自动增量刷新
```

In [ ]:
# 普通视图（每次查询都重新计算）
con.execute("""
CREATE OR REPLACE VIEW v_daily_revenue AS
SELECT
    order_date,
    region,
    COUNT(*)              AS order_count,
    ROUND(SUM(amount), 2) AS daily_revenue,
    ROUND(AVG(amount), 2) AS avg_order_value
FROM large_orders
WHERE status = 'completed'
GROUP BY order_date, region
""")

print("普通视图创建完成")
print("查询普通视图（每次执行都重新扫描基础表）:")
start = time.time()
result = con.execute("""
SELECT region, SUM(daily_revenue) AS total
FROM v_daily_revenue
WHERE order_date >= DATE '2023-01-01'
GROUP BY region
""").df()
t_view = (time.time() - start) * 1000
print(f"普通视图查询耗时: {t_view:.2f} ms")
result

In [ ]:
# 模拟物化视图（DuckDB 不支持 MATERIALIZED VIEW DDL，用普通表模拟）
print("=== 创建'物化视图'（预计算结果存储在表中）===")
start = time.time()
con.execute("""
CREATE OR REPLACE TABLE mv_daily_revenue AS
SELECT
    order_date,
    region,
    COUNT(*)              AS order_count,
    ROUND(SUM(amount), 2) AS daily_revenue,
    ROUND(AVG(amount), 2) AS avg_order_value
FROM large_orders
WHERE status = 'completed'
GROUP BY order_date, region
""")
t_build = (time.time() - start) * 1000
print(f"物化视图构建耗时: {t_build:.2f} ms")

mv_rows = con.execute("SELECT COUNT(*) FROM mv_daily_revenue").fetchone()[0]
print(f"物化视图大小: {mv_rows:,} 行")

print("\n查询物化视图（读预计算结果，极快）:")
start = time.time()
result = con.execute("""
SELECT region, SUM(daily_revenue) AS total
FROM mv_daily_revenue
WHERE order_date >= DATE '2023-01-01'
GROUP BY region
""").df()
t_mv = (time.time() - start) * 1000
print(f"物化视图查询耗时: {t_mv:.2f} ms")
print(f"\n性能对比: 普通视图 {t_view:.1f}ms vs 物化视图 {t_mv:.1f}ms")
result

In [ ]:
# PostgreSQL 物化视图 DDL 参考
print("""
PostgreSQL 物化视图参考代码:
============================

-- 创建物化视图
CREATE MATERIALIZED VIEW mv_daily_revenue AS
SELECT
    order_date,
    region,
    COUNT(*)       AS order_count,
    SUM(amount)    AS daily_revenue
FROM orders
WHERE status = 'completed'
GROUP BY order_date, region;

-- 在物化视图上创建索引（加速查询）
CREATE INDEX ON mv_daily_revenue (order_date);
CREATE INDEX ON mv_daily_revenue (region);

-- 完全刷新（锁表）
REFRESH MATERIALIZED VIEW mv_daily_revenue;

-- 并发刷新（不锁表，需要有 UNIQUE 索引）
CREATE UNIQUE INDEX ON mv_daily_revenue (order_date, region);
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_daily_revenue;

-- 物化视图的注意事项:
-- 1. 刷新频率取决于数据新鲜度要求
-- 2. 完全刷新会锁表，影响并发读取
-- 3. 对于大表，可以用 pg_cron 定时刷新
-- 4. 某些数据库（Snowflake/Redshift）支持自动增量刷新
""")

---

## 练习题

### 练习 1: 读懂执行计划

**场景**: 分析以下查询的执行计划，并回答问题：

1. 该查询是全表扫描还是索引扫描？
2. JOIN 使用了哪种连接算法？
3. 哪个步骤预计代价最高？
4. 如何优化这个查询？

In [ ]:
# 练习 1: 分析执行计划
print("分析以下查询的执行计划:")
plan = con.execute("""
EXPLAIN ANALYZE
SELECT
    p.category_name,
    COUNT(*) AS order_count,
    ROUND(SUM(o.amount), 2) AS total_revenue,
    ROUND(AVG(o.amount), 2) AS avg_order_value
FROM large_orders o
JOIN products p ON o.product_category_id = p.category_id
WHERE YEAR(o.order_date) = 2023    -- 注意：函数包裹可能阻止索引利用
  AND o.amount > 2000
GROUP BY p.category_name
HAVING COUNT(*) > 5
ORDER BY total_revenue DESC
""").fetchall()

for row in plan:
    print(row[0])

# TODO: 将上述查询改写为更高效的版本（避免 YEAR() 函数，使用范围条件）
print("\n\nTODO: 改写更高效的版本")
# YOUR CODE HERE

In [ ]:
# 练习 1 参考答案
print("改写后的高效版本（避免函数包裹日期，使用范围条件）:")

# 原始版本（使用 YEAR() 函数）
start = time.time()
con.execute("""
SELECT p.category_name, COUNT(*) AS order_count, SUM(o.amount)
FROM large_orders o
JOIN products p ON o.product_category_id = p.category_id
WHERE YEAR(o.order_date) = 2023 AND o.amount > 2000
GROUP BY p.category_name
""").fetchall()
t_original = (time.time() - start) * 1000

# 改写版本（使用范围条件，允许索引使用）
start = time.time()
result = con.execute("""
SELECT p.category_name, COUNT(*) AS order_count, SUM(o.amount)
FROM large_orders o
JOIN products p ON o.product_category_id = p.category_id
WHERE o.order_date >= DATE '2023-01-01'   -- 改为范围条件
  AND o.order_date < DATE '2024-01-01'    -- 避免函数包裹
  AND o.amount > 2000
GROUP BY p.category_name
""").df()
t_optimized = (time.time() - start) * 1000

print(f"原始查询 (YEAR函数): {t_original:.2f} ms")
print(f"改写查询 (范围条件): {t_optimized:.2f} ms")
result

### 练习 2: 识别性能瓶颈

**场景**: 以下查询在生产中运行极慢（数百秒），请分析原因并给出优化方案。

```sql
-- 慢查询：找出每个客户的最后一次订单金额是否高于他的历史平均
SELECT
    customer_id,
    (SELECT amount FROM large_orders o2
     WHERE o2.customer_id = o1.customer_id
     ORDER BY order_date DESC LIMIT 1) AS last_order_amount,
    (SELECT AVG(amount) FROM large_orders o3
     WHERE o3.customer_id = o1.customer_id) AS avg_amount
FROM (SELECT DISTINCT customer_id FROM large_orders) o1
```

**问题**: 这个查询的问题是什么？请用窗口函数改写它。

In [ ]:
# 练习 2: 识别并修复慢查询

# 慢查询版本（相关子查询，每行扫描一次基础表）
print("慢查询版本（相关子查询 - O(n^2) 复杂度）:")
print("-- 对每个客户执行2次子查询，总共 N_customers * 2 次全表扫描")
print("-- 在小数据集上测试:")

start = time.time()
result_slow = con.execute("""
SELECT
    customer_id,
    (SELECT amount FROM large_orders o2
     WHERE o2.customer_id = o1.customer_id
     ORDER BY order_date DESC LIMIT 1) AS last_order_amount,
    (SELECT ROUND(AVG(amount), 2) FROM large_orders o3
     WHERE o3.customer_id = o1.customer_id) AS avg_amount
FROM (SELECT DISTINCT customer_id FROM large_orders LIMIT 100) o1  -- 只取100个客户
""").df()
t_slow = (time.time() - start) * 1000
print(f"耗时（仅100个客户）: {t_slow:.2f} ms")

# TODO: 用窗口函数改写，消除相关子查询
print("\nTODO: 用窗口函数改写以下查询")

In [ ]:
# 练习 2 参考答案：用窗口函数消除相关子查询
print("优化版本（窗口函数 - O(n) 复杂度）:")
start = time.time()
result_fast = con.execute("""
WITH order_stats AS (
    SELECT
        customer_id,
        amount,
        order_date,
        -- 最后一次订单的金额（对每个 customer_id 分区内最新一行）
        FIRST_VALUE(amount) OVER (
            PARTITION BY customer_id
            ORDER BY order_date DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_order_amount,
        -- 客户历史平均订单金额
        AVG(amount) OVER (
            PARTITION BY customer_id
        ) AS avg_amount,
        -- 为去重标记最新一行
        ROW_NUMBER() OVER (
            PARTITION BY customer_id ORDER BY order_date DESC
        ) AS rn
    FROM large_orders
)
SELECT
    customer_id,
    last_order_amount,
    ROUND(avg_amount, 2) AS avg_amount,
    CASE
        WHEN last_order_amount > avg_amount THEN '高于均值'
        WHEN last_order_amount < avg_amount THEN '低于均值'
        ELSE '等于均值'
    END AS comparison
FROM order_stats
WHERE rn = 1  -- 每个客户只取一行
ORDER BY customer_id
LIMIT 20  -- 仅展示前20行
""").df()
t_fast = (time.time() - start) * 1000
print(f"优化版本耗时（全量数据）: {t_fast:.2f} ms")
result_fast.head(10)

### 练习 3: 索引设计题

**场景**: large_orders 表有以下高频查询，请为每个查询设计最合适的索引：

```sql
-- 查询 A（高频报表）: 按日期范围 + 状态过滤
SELECT * FROM large_orders
WHERE order_date BETWEEN '2023-01-01' AND '2023-12-31'
  AND status = 'completed';

-- 查询 B（客户详情）: 按客户查所有订单
SELECT * FROM large_orders
WHERE customer_id = 12345
ORDER BY order_date DESC;

-- 查询 C（区域统计）: 按区域聚合
SELECT region, SUM(amount)
FROM large_orders
GROUP BY region;
```

In [ ]:
# 练习 3: 索引设计分析
print("""
请分析每个查询应使用什么索引（填写 TODO）:

查询 A: WHERE order_date BETWEEN ... AND status = 'completed'
  建议索引: TODO
  理由: ...

查询 B: WHERE customer_id = 12345 ORDER BY order_date DESC
  建议索引: TODO
  理由: ...

查询 C: GROUP BY region
  建议索引: TODO
  理由: ...
""")

# 验证：比较不同过滤条件的数据分布
analysis = con.execute("""
SELECT
    status,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM large_orders
GROUP BY status
ORDER BY count DESC
""").df()
print("状态分布:")
display(analysis)

region_dist = con.execute("""
SELECT region, COUNT(*) AS count,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM large_orders GROUP BY region ORDER BY count DESC
""").df()
print("区域分布:")
region_dist

In [ ]:
# 练习 3 参考答案
print("""
索引设计分析（参考答案）:
=========================

查询 A: WHERE order_date BETWEEN ... AND status = 'completed'
  建议索引: CREATE INDEX idx_date_status ON large_orders (order_date, status);
  理由:
    - 复合索引，order_date 在前（因为是范围查询，放前面效率高）
    - status 在后（等值查询，可进一步缩小范围）
    - 注意：status 只有3个值（低基数），单独建索引效益低
    - 可考虑覆盖索引: INCLUDE (amount, region)

查询 B: WHERE customer_id = 12345 ORDER BY order_date DESC
  建议索引: CREATE INDEX idx_customer_date ON large_orders (customer_id, order_date DESC);
  理由:
    - customer_id 等值查询，放最前
    - order_date DESC 与 ORDER BY 方向一致，避免额外排序
    - 这是经典的"等值 + 排序"复合索引模式

查询 C: GROUP BY region
  建议: 可能不值得建索引！
  理由:
    - region 只有4个值（极低基数），选择性约25%
    - 索引对低选择性列效益极低，优化器可能选择全表扫描
    - 更好的方案：物化视图预计算区域汇总
    - 如果是 OLAP 场景，列存储（DuckDB/Parquet）比索引更有效

额外建议:
  - 监控慢查询日志，优先优化执行频率高的慢查询
  - 索引会增加 INSERT/UPDATE/DELETE 开销
  - 定期检查并删除未使用的索引
  - 使用 pg_stat_user_indexes 监控索引使用情况（PostgreSQL）
""")

### 练习 4: 物化视图设计

**场景**: 业务需要一个每日定时刷新的月度区域销售报表，查询响应时间要求 < 100ms。
基础表 large_orders 有 5000 万行，当前查询需要 30 秒。

**任务**: 设计物化视图方案，写出
1. 物化视图的 CREATE 语句（假设在 PostgreSQL 中）
2. 刷新策略
3. 需要在物化视图上建什么索引

In [ ]:
# 练习 4: 物化视图设计

# 步骤 1: 创建物化视图（在 DuckDB 中用普通表模拟）
con.execute("""
CREATE OR REPLACE TABLE mv_monthly_region_sales AS
SELECT
    DATE_TRUNC('month', order_date)        AS report_month,
    region,
    product_category_id,
    COUNT(*)                               AS order_count,
    COUNT(DISTINCT customer_id)            AS unique_customers,
    ROUND(SUM(amount), 2)                  AS total_revenue,
    ROUND(AVG(amount), 2)                  AS avg_order_value,
    ROUND(MIN(amount), 2)                  AS min_order_value,
    ROUND(MAX(amount), 2)                  AS max_order_value,
    COUNT(*) FILTER (WHERE status = 'completed')  AS completed_count,
    COUNT(*) FILTER (WHERE status = 'cancelled')  AS cancelled_count
FROM large_orders
GROUP BY DATE_TRUNC('month', order_date), region, product_category_id
""")

mv_size = con.execute("SELECT COUNT(*) FROM mv_monthly_region_sales").fetchone()[0]
print(f"物化视图大小: {mv_size} 行（原始表: 500,000 行）")

# 步骤 2: 查询物化视图
start = time.time()
result = con.execute("""
SELECT
    report_month,
    region,
    SUM(total_revenue) AS revenue,
    SUM(order_count) AS orders
FROM mv_monthly_region_sales
WHERE report_month >= DATE '2023-01-01'
GROUP BY report_month, region
ORDER BY report_month, revenue DESC
""").df()
t_mv = (time.time() - start) * 1000
print(f"物化视图查询耗时: {t_mv:.2f} ms")
result.head(10)

In [ ]:
# PostgreSQL 物化视图完整方案参考
print("""
PostgreSQL 物化视图完整方案:
============================

-- 1. 创建物化视图
CREATE MATERIALIZED VIEW mv_monthly_region_sales AS
SELECT
    DATE_TRUNC('month', order_date)     AS report_month,
    region,
    product_category_id,
    COUNT(*)                            AS order_count,
    COUNT(DISTINCT customer_id)         AS unique_customers,
    SUM(amount)                         AS total_revenue,
    AVG(amount)                         AS avg_order_value
FROM orders
GROUP BY DATE_TRUNC('month', order_date), region, product_category_id
WITH DATA;  -- 立即填充数据

-- 2. 添加支持查询的索引
CREATE UNIQUE INDEX ON mv_monthly_region_sales
    (report_month, region, product_category_id);  -- 唯一索引（支持并发刷新）
CREATE INDEX ON mv_monthly_region_sales (report_month);
CREATE INDEX ON mv_monthly_region_sales (region);

-- 3. 刷新策略
-- 完全刷新（每天 2AM，在 pg_cron 中配置）:
SELECT cron.schedule(
    'refresh-mv-daily',
    '0 2 * * *',  -- 每天凌晨2点
    'REFRESH MATERIALIZED VIEW CONCURRENTLY mv_monthly_region_sales'
);

-- 并发刷新（不锁表，要求有 UNIQUE 索引）:
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_monthly_region_sales;
""")

---

## 复习要点

### 性能调优核心思路

```
1. 测量（Measure）: 先用 EXPLAIN ANALYZE 找出瓶颈，不要猜测
2. 理解（Understand）: 看 actual rows vs estimated rows，找出估算偏差
3. 优化（Optimize）: 按以下顺序考虑优化手段:
   a. 改写查询（消除函数包裹、相关子查询）
   b. 添加合适索引
   c. 更新统计信息（ANALYZE）
   d. 分区裁剪
   e. 物化视图
```

### 索引决策速查

| 场景 | 选择 |
|------|------|
| 等值 + 范围查询 | B-Tree 复合索引 |
| 只有等值查询 | Hash 索引（某些数据库）|
| 低基数列（状态/性别）| Bitmap 索引（OLAP 数仓）|
| 查询列全在索引内 | 覆盖索引（INCLUDE）|
| 等值 + ORDER BY | 复合索引（等值列在前，排序列在后）|

### 面试高频问题

1. "如何找出慢查询" → 慢查询日志 + EXPLAIN ANALYZE，找 Seq Scan 大表、估算偏差大的节点
2. "B-Tree 和 Hash 索引的区别" → B-Tree 支持范围查询，Hash 只支持等值
3. "什么是分区裁剪" → WHERE 包含分区键时，优化器跳过不相关分区
4. "物化视图什么时候用" → 重复计算的复杂聚合查询，允许一定程度的数据滞后
5. "为什么统计信息很重要" → 优化器依赖统计信息选择执行计划，过时的统计会导致次优计划